#### Initialize the libraries, the environment and the client

In [1]:
import pandas as pd
import os
from pprint import pprint

try:
    from epo.tipdata.ops import OPSClient, models, exceptions
except ImportError:
    os.system('pip install epo-tipdata-ops')
    from epo.tipdata.ops import OPSClient, models, exceptions

%load_ext dotenv
%dotenv
%load_ext autoreload
%autoreload 2
    
client = OPSClient(
    key=os.getenv("OPS_KEY"), secret=os.getenv("OPS_SECRET")
)

### 7.6 Register Method

#### 7.6.1 register(reference_type, input, constituents, output_type)
* Provides access to the European Patent Register online service, allowing you to retrieve all publicly available information for published European patent applications and international PCT applications designating the EPO.<br>
* Parameters:<br>
```reference_type (str)```: Type of patent reference (e.g., "publication", "application", or "priority")<br>
```input (Epodoc)```: Patent document identifier (only Epodoc format supported)<br>
```constituents (list[str], optional)```: Additional information to retrieve (e.g., "events", "procedural-steps") - defaults to "biblio" if not specified<br>
```output_type (str)```: Format of the returned data (raw XML or Pandas DataFrame)<br>

#### 7.6.2 Example of making a register retrieval
This example retrieves bibliographic data (biblio) and procedural steps (procedural-steps) for a published European patent application using its document number, providing once a XML output read with pprint() and once a Pandas dataframe.

In [2]:
# Reset to default options if needed
pd.set_option('display.max_colwidth', 200)

df = client.register(
    reference_type='application',
    input=models.Epodoc('EP16708107'), # EP16708107, EP23159292
    constituents=["upp"],  # "biblio", "events", "procedural-steps", "upp"
    output_type='dataframe'
)

# Print the columns of the DataFrame to verify presence
print("DataFrame Columns:", df.columns)

from IPython.display import display, HTML

def highlight_keywords(val):
    if isinstance(val, dict):  # Convert dict to string
        val = str(val)
    elif not isinstance(val, str):  # Handle other non-string types (e.g., NaN, numbers)
        return val
    
    keywords = ["UDFI", "UDEC", "UREG"]
    for kw in keywords:
        val = val.replace(kw, f'<b><mark style="background-color: yellow">{kw}</mark></b>')
    return val

# Apply styling with highlighted keywords using .format() for proper HTML rendering
df_styled = df.style.format({col: highlight_keywords for col in df.columns}) \
                    .set_properties(**{'text-align': 'left'})

# Display the styled DataFrame
display(df_styled)


DataFrame Columns: Index(['ops:world-patent-data|xmlns:ops', 'ops:world-patent-data|xmlns:reg',
       'ops:world-patent-data|xmlns:xlink', 'ops:world-patent-data|xmlns:cpc',
       'ops:world-patent-data|xmlns:cpcdef',
       'ops:world-patent-data|ops:register-search|total-result-count',
       'ops:world-patent-data|ops:register-search|ops:query',
       'ops:world-patent-data|ops:register-search|ops:range',
       'ops:world-patent-data|ops:register-search|reg:register-documents'],
      dtype='object')


#### 7.6.3 Explanation:
1.	We define the document number for the published European patent application.<br>
2.	We create an Epodoc object with the document number.<br>
3.	We call the register method with the following arguments:<br>
- ```reference_type```: Set to "publication" as we're looking for a published application.<br>
- ```input```: The my_epodoc object containing the document number.<br>
- ```constituents```: Set to a list containing "biblio", "events", "procedural-steps" and "upp" to retrieve bibliographic data, event codes, procedural steps and unitary patent information during the grant process. (By default, biblio is retrieved without specifying it).<br>
- ```output_type```: Set to "dataframe" to get the results as a Pandas DataFrame for easier manipulation.<br>
4.	The register method retrieves the data for the European patent register based on the provided document number and format specifications. It then uses _get_content to convert the response to the desired format (DataFrame in this case).<br>

#### Note:
- This example retrieves both bibliographic data, event codes, procedural steps and upp information. You can adjust the constituents argument to retrieve only bibliographic data (["biblio"]), only event codes (["events"]), only procedural steps (["procedural-steps"]) or only Unitary Patent Protection data (["upp"]) as needed.<br>
- Remember that the register method only accepts the Epodoc format for input.<br>
- Refer to the method documentation for details on other potential options like filtering by specific procedural steps.<br>

In [3]:
# Reset to default options if needed
pd.set_option('display.max_colwidth', 200)

df = client.register(
    reference_type='application',
    input=models.Epodoc('EP16708107'), # EP23159292
    constituents=["upp"],  # "biblio", "events", "procedural-steps", "upp"
    output_type='dataframe'
)

# Print the columns of the DataFrame to verify presence
print("DataFrame Columns:", df.columns)

from IPython.display import display, HTML
import re  # Required for regex-based replacements

# Define keyword categories with colors
highlight_rules = {
    r'\b(UDFI|UDEC|UREG)\b': 'yellow',         # Keywords
    r'@\w+': 'lightblue',                       # Tags starting with "@"
    r'\breg:\w+': 'lightgreen',                  # Labels starting with "reg:"
    r'\bEPIDOS\w+': 'orange',                     # Example: Specific terms
    r'\bEVT_UP\w+': 'red',                       # Example: Important notices
    # r'\bOPT-IN\b': 'orange',                     # Example: Specific terms
    # r'\bOPT-OUT\b': 'red',                       # Example: Important notices
    r'\bEP\d{8}\b': 'pink',                    # Example: EP numbers (patent format)
}

def highlight_keywords(val):
    if isinstance(val, dict):  # Convert dict to string
        val = str(val)
    elif not isinstance(val, str):  # Handle other non-string types (e.g., NaN, numbers)
        return val

    # Apply regex-based color formatting
    for pattern, color in highlight_rules.items():
        # val = re.sub(pattern, rf'<mark style="background-color: {color}">\g<0></mark>', val)
        if 'EPIDOS' in pattern or 'EVT_UP' in pattern:
            # Make EPIDOS & EVT_UP bold + colored
            val = re.sub(pattern, rf'<mark style="background-color: {color}"><b>\g<0></b></mark>', val)
        else:
            # Apply only color for other matches
            val = re.sub(pattern, rf'<mark style="background-color: {color}">\g<0></mark>', val)

    return val

# Convert DataFrame to HTML with formatted text
html_table = df.to_html(escape=False, formatters={col: highlight_keywords for col in df.columns})

# Display the styled DataFrame
display(HTML(html_table))


DataFrame Columns: Index(['ops:world-patent-data|xmlns:ops', 'ops:world-patent-data|xmlns:reg',
       'ops:world-patent-data|xmlns:xlink', 'ops:world-patent-data|xmlns:cpc',
       'ops:world-patent-data|xmlns:cpcdef',
       'ops:world-patent-data|ops:register-search|total-result-count',
       'ops:world-patent-data|ops:register-search|ops:query',
       'ops:world-patent-data|ops:register-search|ops:range',
       'ops:world-patent-data|ops:register-search|reg:register-documents'],
      dtype='object')


In [4]:
# Reset to default options if needed
pd.set_option('display.max_colwidth', 200)

df = client.register(
    reference_type='application',
    input=models.Epodoc('EP23159292'), # EP23159292
    constituents=["upp"],  # "biblio", "events", "procedural-steps", "upp"
    output_type='dataframe'
)

# Print the columns of the DataFrame to verify presence
print("DataFrame Columns:", df.columns)

from IPython.display import display, HTML

def highlight_keywords(val):
    if isinstance(val, dict):  # Convert dict to string
        val = str(val)
    elif not isinstance(val, str):  # Handle other non-string types (e.g., NaN, numbers)
        return val

    # Define color mappings for different tags and keywords
    color_map = {
        "UDFI": "yellow",  # Keyword
        "UDEC": "yellow",  # Keyword
        "UREG": "yellow",  # Keyword
        "@": "blue",  # Tag starting with '@'
        "reg:": "green",  # Label starting with 'reg:'
    }

    # Highlight keywords, tags, and labels with colors
    for keyword, color in color_map.items():
        if keyword in val:
            val = val.replace(keyword, f'<mark style="background-color: {color}">{keyword}</mark>')

    return val

# Apply styling with highlighted keywords using .format() for proper HTML rendering
df_styled = df.style.format({col: highlight_keywords for col in df.columns}) \
                    .set_properties(**{'text-align': 'left'})

# Display the styled DataFrame
display(df_styled)


DataFrame Columns: Index(['ops:world-patent-data|xmlns:ops', 'ops:world-patent-data|xmlns:reg',
       'ops:world-patent-data|xmlns:xlink', 'ops:world-patent-data|xmlns:cpc',
       'ops:world-patent-data|xmlns:cpcdef',
       'ops:world-patent-data|ops:register-search|total-result-count',
       'ops:world-patent-data|ops:register-search|ops:query',
       'ops:world-patent-data|ops:register-search|ops:range',
       'ops:world-patent-data|ops:register-search|reg:register-documents'],
      dtype='object')


#### 7.6.3.1 Data Normalization and Extraction from Patent Register API Response
* This cell demonstrates how to interact with a patent register client to retrieve and normalize data related to a specific patent application. 
* The process begins with registering a patent application and specifying the desired constituents (bibliographic data, events, and procedural steps) while requesting the output as a DataFrame.
* The code then extracts a specific column containing nested data, checks if each entry is a dictionary, and normalizes this nested structure using pandas.json_normalize(). It filters the resulting DataFrame to focus on columns that contain @ attributes, which often represent metadata in JSON structures. Finally, the structured output is displayed, and the available columns in the DataFrame are printed for further analysis. 

In [5]:
import pandas as pd
import json

content = client.register(
    reference_type='application',
    input=models.Epodoc('EP09164213'), # EP99203729
    constituents=["biblio", "events", "procedural-steps", "upp"], # "biblio", "events", "procedural-steps", "upp"
    output_type='dataframe'
)
display(content)

# Display only the last column with the specified name
last_column_name = 'ops:world-patent-data|ops:register-search|reg:register-documents'

# Check if the expected column exists
if last_column_name in content.columns:
    # Try converting to dictionary if it's a string
    content[last_column_name] = content[last_column_name].apply(lambda x: json.loads(x) if isinstance(x, str) else x)

    # Extract the first row's data (assuming it's a dictionary)
    nested_data = content[last_column_name].iloc[0] if not content.empty else {}

    # Check if nested_data is a dictionary
    if isinstance(nested_data, dict):
        print("Successfully extracted nested JSON data.")
    else:
        print("Error: Extracted data is not a dictionary.")
else:
    print(f"Column '{last_column_name}' not found.")

# Normalize the nested dictionary
flattened_data = pd.json_normalize(nested_data)
print("flattened_data.columns:", flattened_data.columns)

# Display the flattened data focusing on `@` or 'reg:' attributes
# Optionally, filter columns to only include those with `@` or 'reg:' in their names
at_columns = [col for col in flattened_data.columns if '@' or 'reg:' in col]
structured_output = flattened_data[at_columns]

# Display the structured output
display(structured_output)

# Check available columns
print("Available Columns:", structured_output.columns) # .tolist()

!pip install xmltodict
import xmltodict

xml_data = client.register(
    reference_type='application',
    input=models.Epodoc('EP09164213'), # EP99203729
    constituents=["biblio", "events", "procedural-steps", "upp"],
    output_type='raw'
)

# Convert XML to a dictionary
json_data = json.loads(json.dumps(xmltodict.parse(xml_data), indent=4))

# Print JSON
print(json.dumps(json_data, indent=10))

,ops:world-patent-data|xmlns:ops,ops:world-patent-data|xmlns:reg,ops:world-patent-data|xmlns:xlink,ops:world-patent-data|xmlns:cpc,ops:world-patent-data|xmlns:cpcdef,ops:world-patent-data|ops:register-search|total-result-count,ops:world-patent-data|ops:register-search|ops:query,ops:world-patent-data|ops:register-search|ops:range,ops:world-patent-data|ops:register-search|reg:register-documents
0,http://ops.epo.org,http://www.epo.org/register,http://www.w3.org/1999/xlink,http://www.epo.org/cpcexport,http://www.epo.org/cpcdefinition,1,"{'@syntax': 'CQL', '#text': 'application=EP09164213'}","{'@begin': '1', '@end': '1'}","{'@produced-by': 'RO', 'reg:register-document': {'@date-produced': '20250218', '@dtd-version': '1.3.3', '@lang': 'en', '@produced-by': 'RO', '@status': 'No opposition filed within time limit', 're..."


Successfully extracted nested JSON data.
flattened_data.columns: Index(['@produced-by', 'reg:register-document.@date-produced',
       'reg:register-document.@dtd-version', 'reg:register-document.@lang',
       'reg:register-document.@produced-by', 'reg:register-document.@status',
       'reg:register-document.reg:ep-patent-statuses.reg:ep-patent-status.@change-date',
       'reg:register-document.reg:ep-patent-statuses.reg:ep-patent-status.@status-code',
       'reg:register-document.reg:ep-patent-statuses.reg:ep-patent-status.#text',
       'reg:register-document.reg:bibliographic-data.@country',
       'reg:register-document.reg:bibliographic-data.@id',
       'reg:register-document.reg:bibliographic-data.@lang',
       'reg:register-document.reg:bibliographic-data.@status',
       'reg:register-document.reg:bibliographic-data.reg:publication-reference',
       'reg:register-document.reg:bibliographic-data.reg:classifications-ipcr',
       'reg:register-document.reg:bibliographic-da

,@produced-by,reg:register-document.@date-produced,reg:register-document.@dtd-version,reg:register-document.@lang,reg:register-document.@produced-by,reg:register-document.@status,reg:register-document.reg:ep-patent-statuses.reg:ep-patent-status.@change-date,reg:register-document.reg:ep-patent-statuses.reg:ep-patent-status.@status-code,reg:register-document.reg:ep-patent-statuses.reg:ep-patent-status.#text,reg:register-document.reg:bibliographic-data.@country,...,reg:register-document.reg:bibliographic-data.reg:dates-rights-effective.reg:request-for-examination.reg:date,reg:register-document.reg:bibliographic-data.reg:dates-rights-effective.reg:first-examination-report-despatched.reg:date,reg:register-document.reg:bibliographic-data.reg:references-cited.reg:citation,reg:register-document.reg:bibliographic-data.reg:related-documents.reg:division,reg:register-document.reg:bibliographic-data.reg:opposition-data.@change-date,reg:register-document.reg:bibliographic-data.reg:opposition-data.@change-gazette-num,reg:register-document.reg:bibliographic-data.reg:opposition-data.reg:opposition-not-filed.reg:date,reg:register-document.reg:bibliographic-data.reg:search-reports-information.reg:search-report-information,reg:register-document.reg:procedural-data.reg:procedural-step,reg:register-document.reg:events-data
0,RO,20250218,1.3.3,en,RO,No opposition filed within time limit,,7,No opposition filed within time limit,EP,...,20090630,20091229,"[{'@id': 'cit_15778947', '@cited-phase': 'search', '@office': 'EP', 'reg:patcit': {'@url': 'https://worldwide.espacenet.com/patent/search?q=pn%3DJPH07298311', 'reg:document-id': {'reg:country': 'J...","[{'reg:relation': {'reg:parent-doc': {'reg:document-id': [{'@document-id-type': 'application number', 'reg:country': 'EP', 'reg:doc-number': '20060011611', 'reg:kind': 'D'}, {'@document-id-type': ...",20131129,2014/01,20131024,"[{'@office': 'EP', '@search-type': 'national', '@declaration-of-no-search': 'not-determined', '@change-gazette-num': '2009/45', 'reg:search-report-publication': {'reg:document-id': {'reg:country':...","[{'@id': 'RENEWAL_5085487', '@procedure-step-phase': 'undefined', 'reg:procedural-step-code': 'RFEE', 'reg:procedural-step-text': [{'@step-text-type': 'STEP_DESCRIPTION', '#text': 'Renewal fee pay...","[{'reg:dossier-event': {'@id': 'EVT_39674808', '@event-type': 'new', 'reg:event-date': {'reg:date': '20131129'}, 'reg:event-code': '0009261', 'reg:event-text': {'@event-text-type': 'DESCRIPTION', ..."


Available Columns: Index(['@produced-by', 'reg:register-document.@date-produced',
       'reg:register-document.@dtd-version', 'reg:register-document.@lang',
       'reg:register-document.@produced-by', 'reg:register-document.@status',
       'reg:register-document.reg:ep-patent-statuses.reg:ep-patent-status.@change-date',
       'reg:register-document.reg:ep-patent-statuses.reg:ep-patent-status.@status-code',
       'reg:register-document.reg:ep-patent-statuses.reg:ep-patent-status.#text',
       'reg:register-document.reg:bibliographic-data.@country',
       'reg:register-document.reg:bibliographic-data.@id',
       'reg:register-document.reg:bibliographic-data.@lang',
       'reg:register-document.reg:bibliographic-data.@status',
       'reg:register-document.reg:bibliographic-data.reg:publication-reference',
       'reg:register-document.reg:bibliographic-data.reg:classifications-ipcr',
       'reg:register-document.reg:bibliographic-data.reg:application-reference.@change-gazette-n

#### 7.6.3.2 Exploring and Visualizing Patent Register Data with Pretty-Print Functionality
This cell defines a function called pretty_print, which is designed to recursively display nested JSON-like data structures in a human-readable format. It checks whether the input data is a dictionary, a list, or a primitive value, and formats the output with appropriate indentation based on the level of nesting.

In [6]:
import json

# Function to pretty-print nested data with indentation
def pretty_print(data, indent=0):
    if isinstance(data, dict):
        for key, value in data.items():
            print('  ' * indent + str(key) + ":")
            pretty_print(value, indent + 1)
    elif isinstance(data, list):
        for i, item in enumerate(data):
            print('  ' * indent + f"Item {i}:")
            pretty_print(item, indent + 1)
    else:
        print('  ' * indent + str(data))

# Extract the nested dictionary from the column
nested_data = content[last_column_name][0]  # Assuming one row; adjust as needed

# Pretty-print the nested structure
pretty_print(nested_data)

# print("Raw nested_data:", repr(nested_data))

@produced-by:
  RO
reg:register-document:
  @date-produced:
    20250218
  @dtd-version:
    1.3.3
  @lang:
    en
  @produced-by:
    RO
  @status:
    No opposition filed within time limit
  reg:ep-patent-statuses:
    reg:ep-patent-status:
      @change-date:
        
      @status-code:
        7
      #text:
        No opposition filed within time limit
  reg:bibliographic-data:
    @country:
      EP
    @id:
      EP09164213P
    @lang:
      en
    @status:
      No opposition filed within time limit
    reg:publication-reference:
      Item 0:
        @change-gazette-num:
          2009/38
        reg:document-id:
          @lang:
            en
          reg:country:
            EP
          reg:doc-number:
            2101496
          reg:kind:
            A2
          reg:date:
            20090916
      Item 1:
        @change-gazette-num:
          2009/45
        reg:document-id:
          reg:country:
            EP
          reg:doc-number:
            2101496
       

In [7]:
import json

# Function to pretty-print nested data with indentation
def pretty_print(data, indent=0):
    if isinstance(data, dict):
        for key, value in data.items():
            print('  ' * indent + str(key) + ":")
            pretty_print(value, indent + 1)
    elif isinstance(data, list):
        for i, item in enumerate(data):
            print('  ' * indent + f"Item {i}:")
            pretty_print(item, indent + 1)
    else:
        print('  ' * indent + str(data))

# Extract the nested dictionary from the column
nested_data = content[last_column_name][0]  # Assuming one row; adjust as needed

# Pretty-print the nested structure
pretty_print(nested_data)

# print("Raw nested_data:", repr(nested_data))

@produced-by:
  RO
reg:register-document:
  @date-produced:
    20250218
  @dtd-version:
    1.3.3
  @lang:
    en
  @produced-by:
    RO
  @status:
    No opposition filed within time limit
  reg:ep-patent-statuses:
    reg:ep-patent-status:
      @change-date:
        
      @status-code:
        7
      #text:
        No opposition filed within time limit
  reg:bibliographic-data:
    @country:
      EP
    @id:
      EP09164213P
    @lang:
      en
    @status:
      No opposition filed within time limit
    reg:publication-reference:
      Item 0:
        @change-gazette-num:
          2009/38
        reg:document-id:
          @lang:
            en
          reg:country:
            EP
          reg:doc-number:
            2101496
          reg:kind:
            A2
          reg:date:
            20090916
      Item 1:
        @change-gazette-num:
          2009/45
        reg:document-id:
          reg:country:
            EP
          reg:doc-number:
            2101496
       

#### 7.6.3.3 Extracting and Visualizing Patent Document Metadata Using AnyTree
* This cell focuses on extracting specific metadata from a structured DataFrame containing patent register data and visualizing it in a hierarchical tree format using the anytree library. The process begins by checking if the structured_output DataFrame is not empty, ensuring that there is data to process. It then extracts critical attributes from the first row of the DataFrame, including the producer of the document, the date produced, the status of the document in the register, and the document number.
* To handle potential missing data, the code uses the .get() method, providing default values when specific columns may not exist.
* The second part of the code constructs a tree structure where the root node represents the "register-document," and its children nodes encapsulate the extracted metadata attributes. This hierarchical visualization makes it easier to understand the relationships between different pieces of data associated with the patent register.
* Finally, the code uses the RenderTree function to display the constructed tree in a readable format. If the structured_output is empty, it prints a message indicating that no data is available. This approach enhances data interpretation and facilitates a clearer understanding of patent document metadata by leveraging visual hierarchy.

In [8]:
!pip install anytree
from anytree import Node, RenderTree

# Assuming the first row contains the data we want to extract
if not structured_output.empty:
    if 'reg:register-document.@produced-by' in structured_output.columns:
        first_row = structured_output.iloc[0]
        # Proceed with your logic
    else:
        print("'@produced-by' not found in structured_output.")


    # Use .get() to avoid KeyError and check for column existence
    produced_by = first_row.get('reg:register-document.@produced-by', 'Unknown')  # e.g., "RO"
    date_produced = first_row.get('reg:register-document.@date-produced', 'Unknown')  # e.g., "20241010"
    status = first_row.get('reg:register-document.@status', 'Unknown')  # e.g., "No opposition filed within time limit"
    doc_number = first_row.get('reg:register-document.reg:bibliographic-data.@id', 'Unknown')  # Assuming this column exists

    # Second part: Create nodes for the tree
    root = Node("register-document")
    Node(f"@produced-by: {produced_by}", parent=root)
    Node(f"@date-produced: {date_produced}", parent=root)
    Node(f"@status: {status}", parent=root)
    bibliographic = Node("reg:bibliographic-data", parent=root)
    Node(f"Document Number: {doc_number}", parent=bibliographic)

    # Render tree
    for pre, fill, node in RenderTree(root):
        print(f"{pre}{node.name}")
else:
    print("No data available in structured_output.")

register-document
├── @produced-by: RO
├── @date-produced: 20250218
├── @status: No opposition filed within time limit
└── reg:bibliographic-data
    └── Document Number: EP09164213P


In [9]:
from anytree import Node, RenderTree

def build_tree(data, parent):
    """Recursively build anytree structure from nested dictionary data."""
    if isinstance(data, dict):
        for key, value in data.items():
            child = Node(f"{key}: {value if not isinstance(value, (dict, list)) else ''}", parent=parent)
            build_tree(value, child)  # Recursively add children
    elif isinstance(data, list):
        for i, item in enumerate(data):
            child = Node(f"Item {i}", parent=parent)
            build_tree(item, child)
    else:
        Node(str(data), parent=parent)  # Leaf nodes (e.g., plain values)

# Create root node
root = Node("register-document")

# Build tree from nested_data
build_tree(nested_data, root)

# Render tree
for pre, fill, node in RenderTree(root):
    print(f"{pre}{node.name}")

register-document
├── @produced-by: RO
│   └── RO
└── reg:register-document: 
    ├── @date-produced: 20250218
    │   └── 20250218
    ├── @dtd-version: 1.3.3
    │   └── 1.3.3
    ├── @lang: en
    │   └── en
    ├── @produced-by: RO
    │   └── RO
    ├── @status: No opposition filed within time limit
    │   └── No opposition filed within time limit
    ├── reg:ep-patent-statuses: 
    │   └── reg:ep-patent-status: 
    │       ├── @change-date: 
    │       │   └── 
    │       ├── @status-code: 7
    │       │   └── 7
    │       └── #text: No opposition filed within time limit
    │           └── No opposition filed within time limit
    ├── reg:bibliographic-data: 
    │   ├── @country: EP
    │   │   └── EP
    │   ├── @id: EP09164213P
    │   │   └── EP09164213P
    │   ├── @lang: en
    │   │   └── en
    │   ├── @status: No opposition filed within time limit
    │   │   └── No opposition filed within time limit
    │   ├── reg:publication-reference: 
    │   │   ├── Item 0


#### 7.6.3.4 Analyzing the Structure of Register Data from a Pandas Series
* This cell is designed to analyze the contents of a Pandas Series called nested_data, which is assumed to contain nested dictionary-like structures related to patent register documents. The script first checks if nested_data is indeed a Series and that it is not empty.
* If these conditions are met, the code extracts the first item from the Series. It then verifies whether this item is a dictionary and whether it contains a specific key: 'reg:register-document'. If the key exists, the script prints the keys found within that dictionary, providing insights into the structure and available attributes of the register data. If the expected key is absent, or if the first item is not a dictionary, the code outputs an appropriate message indicating the issue.
* This approach is useful for gaining a quick overview of the data structure, helping users understand the organization of the nested register data, before setting the stage for more in-depth analyses and visualizations.

In [10]:
import json
import ast
# print(nested_data)
# Extract the first item from the Series
if isinstance(nested_data, pd.Series):
    nested_data = nested_data.iloc[0]  # Get the first item

# Try different methods to convert string to dictionary
if isinstance(nested_data, str):
    try:
        # First, try decoding JSON normally
        nested_data = json.loads(nested_data)
    except json.JSONDecodeError:
        try:
            # If JSON decoding fails, try using ast.literal_eval()
            nested_data = ast.literal_eval(nested_data)  
        except (SyntaxError, ValueError):
            print("Error: Could not decode JSON or evaluate string. Check the format of nested_data.")

# Now check if it's a dictionary before proceeding
if isinstance(nested_data, dict):
    if "reg:register-document" in nested_data:
        print("Keys in 'reg:register-document':", nested_data["reg:register-document"].keys())
    else:
        print("'reg:register-document' not found in the dictionary.")
else:
    print("The extracted item is not a dictionary.")


Keys in 'reg:register-document': dict_keys(['@date-produced', '@dtd-version', '@lang', '@produced-by', '@status', 'reg:ep-patent-statuses', 'reg:bibliographic-data', 'reg:procedural-data', 'reg:events-data'])


#### 7.6.3.5 Inspecting and Printing the Structure of Nested Data in a Pandas Series
This code aims to explore and understand the layout and structure of nested_data, which is a Pandas Series.
This inspection is a critical step in data analysis, especially when working with nested or complex structures.

In [11]:
# Print the entire nested_data structure to understand its layout
import json

# Check the type of nested_data
print(type(nested_data))

# If nested_data is a Series, print its contents
# print("nested_data:", nested_data)

<class 'dict'>


#### 7.6.3.6 Handling Nested JSON Data in a Pandas Series for Patent Publication and Event Information
* The cell processes a nested JSON structure stored within a Pandas Series. It extracts publication references and event information from a patent's bibliographic and status data. The code includes error handling to accommodate both cases where the patent status data ('reg:ep-patent-status') is either a list or a single dictionary. Key steps include:
* Extracting bibliographic data: The code normalizes and displays the publication references using pd.json_normalize() if found within 'reg:bibliographic-data'.
* Handling events: It checks if the event data ('reg:ep-patent-status') is a list or a dictionary and processes it accordingly, ensuring compatibility with pd.json_normalize().
* Displays results: The publication references and event details are printed in a DataFrame format to allow easy inspection.

In [12]:
!pip install matplotlib seaborn

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming nested_data is defined as a Pandas Series with dictionaries

# Check if nested_data is a Series and not empty
if isinstance(nested_data, pd.Series) and not nested_data.empty:
    for idx, item in nested_data.items():
        print(f"\nProcessing item at index {idx}:")
        # print(f"Item content: {item}")
        print(f"(type(item): {type(item)})")
        
        if isinstance(item, str):
            print(f"Item at index {idx} is a string ('{item}') and will be skipped.")
            continue
        
        if isinstance(item, dict):
            if 'reg:bibliographic-data' in item:
                bibliographic_data = item['reg:bibliographic-data']

                # Normalize publication references
                publication_references = pd.json_normalize(
                    bibliographic_data,
                    record_path=['reg:publication-reference'],
                    meta=['@produced-by', '@date-produced'],
                    errors='ignore'
                )

                # Visualization: Bar chart for document kinds
                document_kinds = publication_references['reg:document-id.reg:kind'].value_counts().reset_index()
                document_kinds.columns = ['Document Kind', 'Count']

                plt.figure(figsize=(10, 6))
                sns.barplot(x='Document Kind', y='Count', data=document_kinds, color='royalblue')  # Use a single color
                plt.title('Number of Publication References by Document Kind')
                plt.xlabel('Document Kind')
                plt.ylabel('Count')
                plt.xticks(rotation=45)
                plt.show()

            if 'reg:ep-patent-statuses' in item:
                ep_patent_statuses = item['reg:ep-patent-statuses']
                ep_patent_status = ep_patent_statuses.get('reg:ep-patent-status')

                if isinstance(ep_patent_status, list):
                    events_data = pd.json_normalize(
                        ep_patent_statuses,
                        record_path=['reg:ep-patent-status'],
                        meta=['@status-code', '@change-date'],
                        errors='ignore'
                    )
                elif isinstance(ep_patent_status, dict):
                    events_data = pd.json_normalize(
                        [ep_patent_status],
                        meta=['@status-code', '@change-date'],
                        errors='ignore'
                    )

                if not events_data.empty:
                    print("\nEvents Data Columns:", events_data.columns.tolist())
                    print(events_data)

                    # Count the occurrences of each status code
                    status_count = events_data['@status-code'].value_counts().reset_index()
                    status_count.columns = ['Status Code', 'Count']

                    plt.figure(figsize=(10, 6))
                    bars = sns.barplot(x='Status Code', y='Count', data=status_count, color='royalblue')  # Use a single color

                    plt.title('Count of Patent Status Codes')
                    plt.xlabel('Status Code')
                    plt.ylabel('Count')
                    plt.xticks(rotation=45)

                    # Manual legend creation
                    handles = [plt.Line2D([0], [0], marker='o', color='w', label=status, markerfacecolor='royalblue', markersize=10)
                               for status in status_count['Status Code']]
                    plt.legend(handles=handles, title='Status Code', loc='upper right', bbox_to_anchor=(1.15, 1))

                    plt.show()
                else:
                    print("No events data available.")
            else:
                print("'reg:ep-patent-statuses' not found in the current item.")
        else:
            print(f"Item at index {idx} is not a dictionary and will be skipped.")
else:
    print("nested_data is empty or not a Pandas Series.")


nested_data is empty or not a Pandas Series.
